# 人脸识别

欢迎！在这个作业中，你将构建一个人脸识别系统。这里提出的许多想法都来自 [FaceNet](https://arxiv.org/pdf/1503.03832.pdf)。在讲座中，你还遇到了[DeepFace](https://research.fb.com/wp-content/uploads/2016/11/deepface-closing-the-gap-to-human-level-performance-in-face-verification.pdf)。

人脸识别问题通常属于以下两类之一： 

**人脸验证** “这是被认领的人吗？”例如，在某些机场，你可以通过让系统扫描你的护照来通过海关，然后验证你（携带护照的人）是否是正确的人。使用人脸解锁的手机也使用人脸验证。这是一个1:1匹配问题。

**人脸识别**回答“这个人是谁？”。例如，课程展示了一段百度员工通过人脸识别进入办公室的[视频](https://www.youtube.com/watch?v=wr4rx0Spihs)，员工无需另行提供身份信息。这是一个 1:K 匹配问题。

FaceNet 学习一个将人脸图像编码为 128 个数字的向量的神经网络。通过比较两个这样的向量，你可以确定两张图片是否是同一个人。

读完本作业，你将能够： 

* 人脸识别和人脸验证的区别
* 实现一次性学习来解决人脸识别问题
* 应用三元组损失函数学习人脸识别背景下的网络参数
* 解释如何将人脸识别视为二元分类问题
* 使用预训练模型将人脸图像映射为 128 维编码
* 使用这些编码进行人脸验证和人脸识别

**频道最后表示法**

对于此作业，你将使用预训练模型，该模型使用“通道最后”约定来表示 ConvNet 激活值，如讲座期间和之前的编程作业中所使用的那样。

换句话说，一批图像的形状为 $(m, n_H, n_W, n_C)$。 

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/DS4yP/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [1 - 依赖库](#1)
- [2 - 朴素人脸验证](#2)
- [3 - 将人脸图像编码为 128 维向量](#3)
    - [3.1 - 使用 ConvNet 计算编码](#3-1)
    - [3.2 - 三元组损失](#3-2)
        - [练习 1 - triplet_loss](#ex-1)
- [4 - 加载预训练模型](#4)
- [5 - 应用模型](#5)
    - [5.1 - 人脸验证](#5-1)
        - [练习 2 - verify](#ex-2)
    - [5.2 - 人脸识别](#5-2)
        - [练习 3 - who_is_it](#ex-3)
- [6 - 参考文献](#6)

<a name='1'></a>
## 1 - 依赖库

继续运行下面的单元格以导入你需要的包。

In [ ]:
### v1.1

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, ZeroPadding2D, Activation, Input, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import MaxPooling2D, AveragePooling2D
from tensorflow.keras.layers import Concatenate
from tensorflow.keras.layers import Lambda, Flatten, Dense
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K
K.set_image_data_format('channels_last')
import os
import numpy as np
from numpy import genfromtxt
import pandas as pd
import tensorflow as tf
import PIL

%matplotlib inline
%load_ext autoreload
%autoreload 2

<a name='2'></a>
## 2 - 朴素人脸验证

在人脸验证中，你会得到两张图像，你必须确定它们是否是同一个人。最简单的方法是逐像素比较两个图像。如果原始图像之间的距离低于所选阈值，则可能是同一个人！

<img src="images/pixel_comparison.png" style="width:380px;height:150px;">
<caption><center> <u> <font color='purple'> <b>图1</b> </u></center></caption>

当然，这种算法的性能很差，因为像素值由于光照变化、人脸方向、头部位置的微小变化等而发生巨大变化。

你将看到，你可以学习编码 $f(img)$，而不是使用原始图像。

通过对每张图像使用编码，逐元素比较可以更准确地判断两张图片是否属于同一个人。

<a name='3'></a>
## 3 - 将人脸图像编码为 128 维向量

<a name='3-1'></a>
### 3.1 - 使用 ConvNet 计算编码

FaceNet 模型需要大量数据和很长时间来训练。因此，按照应用深度学习的常见做法，你将加载其他人已经训练过的权重。网络架构遵循 [塞格迪*等人*..](https://arxiv.org/abs/1409.4842) 的 Inception 模型。我们已经为你提供了 Inception 网络实现，你可以在文件 `inception_blocks_v2.py` 中找到它，以仔细了解它是如何实现的。  

*热门提示：* 转到此 Notebook 顶部的“文件->打开...”。这将打开包含 `.py` 文件的文件目录。

需要注意的关键事项是：

- 该网络使用 160x160 维 RGB 图像作为输入。具体来说，一张人脸图像（或一批 $m$ 人脸图像）作为形状为 $(m, n_H, n_W, n_C) = (m, 160, 160, 3)$ 的张量
- 输入图像的原始形状为 96x96，因此，你需要将它们缩放到 160x160。这是在 `img_to_encoding()` 函数中完成的。
- 输出是形状为 $(m, 128)$ 的矩阵，它将每个输入人脸图像编码为 128 维向量

运行下面的单元格来创建面部图像的模型！

In [ ]:
from tensorflow.keras.models import model_from_json

json_file = open('keras-facenet-h5/model.json', 'r')
loaded_model_json = json_file.read()
json_file.close()
model = model_from_json(loaded_model_json)
model.load_weights('keras-facenet-h5/model.h5')

现在总结一下输入和输出形状： 

In [ ]:
print(model.inputs)
print(model.outputs)

通过使用 128 神经元全连接层作为最后一层，模型确保输出是大小为 128 的编码向量。然后，你可以使用编码来比较两个人脸图像，如下所示：

<img src="images/distance_kiank.png\" style="width:680px;height:250px;">
<caption><center> <u> <font color='purple'> <b>图2：</b> <br> </u> <font color='purple'>通过计算两次编码之间的距离和阈值化，可以判断两张图片是否代表同一个人</center></caption>

因此，如果满足以下条件，编码就是一个好的编码：

- 同一个人的两张图像的编码非常相似。
- 不同人的两张图像的编码有很大不同。

三元组损失函数形式化了这一点，并尝试将同一个人的两个图像（Anchor 和 Positive）的编码“推”得更近，同时“拉”不同人的两个图像（Anchor、Negative）的编码进一步分开。
    
<img src="images/triplet_comparison.png" style="width:280px;height:150px;"><br>
<caption><center> <u> <font color='purple'> <b>图 3：</b> <br> </u> <font color='purple'>下一节，你将从左到右调用图片：锚点（A），正（P），负（N）</center></caption>

<a name='3-2'></a>
### 3.2 - 三元组损失

**重要提示**：由于你使用的是预训练模型，因此你实际上不需要在此作业中实现三元组损失函数。 *但是*，三元组损失是人脸识别算法的主要成分，你需要知道如何使用它来训练你自己的 FaceNet 模型，以及其他类型的图像相似性问题。因此，你将在下面实现它，以获取乐趣和启发。 :) 

对于图像 $x$，其编码表示为 $f(x)$，其中 $f$ 是神经网络计算出的函数。

<img src="images/f_x.png" style="width:380px;height:150px;">

训练将使用三组图像 $(A, P, N)$：

- A 是“锚”图像——一个人的照片。
- P 是“正”图像——与锚图像同一个人的照片。
- N 是“负片”图像——与锚图像不同的人的照片。

这些三元组是从训练数据集中挑选的。 $(A^{(i)}, P^{(i)}, N^{(i)})$ 在此用于表示第 $i$ 个训练样本。

你想要确保个体的图像 $A^{(i)}$ 与正图像 $P^{(i)}$ 的距离比与负样本图像 $N^{(i)}$ 的距离至少小一些：$\alpha$：

$$
|| f\left(A^{(i)}\right)-f\left(P^{(i)}\right)||_{2}^{2}+\alpha<|| f\left(A^{(i)}\right)-f\left(N^{(i)}\right)||_{2}^{2}
$$


因此，你希望最小化以下“三元组代价”：

$$\mathcal{J} = \sum^{m}_{i=1} \large[ \small \underbrace{\mid \mid f(A^{(i)}) - f(P^{(i)}) \mid \mid_2^2}_\text{(1)} - \underbrace{\mid \mid f(A^{(i)}) - f(N^{(i)}) \mid \mid_2^2}_\text{(2)} + \alpha \large ] \small_+ \tag{3}$$
这里，符号“$[z]_+$”用于表示$max(z,0)$。

**注释**：

- 项（1）是给定三元组的锚点“A”和正“P”之间的平方距离；你希望它很小。
- 第（2）项是给定三元组中锚点样本 A 与负样本 N 的编码距离平方，我们希望它较大。因此，该项前面带有负号：最小化其相反数就相当于最大化该项。
- $\alpha$ 称为保证金。这是你手动选择的超参数。你将使用 $\alpha = 0.2$。

大多数实现还会重新调整编码向量以使 L2 范数等于 1（即$\mid \mid f(img)\mid \mid_2$=1）；在作业中你不必担心这一点。

<a name='ex-1'></a>
### 练习 1 - triplet_loss

实现公式（3）定义的三元组损失。以下是 4 个步骤：

1. 计算“anchor”和“positive”的编码之间的距离：$\mid \mid f(A^{(i)}) - f(P^{(i)}) \mid \mid_2^2$
2. 计算“anchor”和“negative”的编码之间的距离：$\mid \mid f(A^{(i)}) - f(N^{(i)}) \mid \mid_2^2$
3. 计算训练样本的公式：$ \mid \mid f(A^{(i)}) - f(P^{(i)}) \mid \mid_2^2 - \mid \mid f(A^{(i)}) - f(N^{(i)}) \mid \mid_2^2 + \alpha$
4. 通过取最大值为零并对训练样本:$$\mathcal{J} = \sum^{m}_{i=1} \large[ \small \mid \mid f(A^{(i)}) - f(P^{(i)}) \mid \mid_2^2 - \mid \mid f(A^{(i)}) - f(N^{(i)}) \mid \mid_2^2+ \alpha \large ] \small_+ \tag{3}$$ 求和来计算完整公式

*提示*：

- 有用的函数：`tf.reduce_sum()`、`tf.square()`、`tf.subtract()`、`tf.add()`、`tf.maximum()`。

- 对于步骤 1 和 2，对 $\mid \mid f(A^{(i)}) - f(P^{(i)}) \mid \mid_2^2$ 和 $\mid \mid     f(A^{(i)}) - f(N^{(i)}) \mid \mid_2^2$ 的条目求和。

- 对于步骤 4，你将对训练样本求和。

*其他提示*：

- 回想一下，L2 范数的平方是差平方和：$||x - y||_{2}^{2} = \sum_{i=1}^{N}(x_{i} - y_{i})^{2}$

- 请注意，锚、正编码和负编码的形状为 (*m*,128)，其中 *m* 是训练样本的数量，128 是用于对单个示例进行编码的元素数量。

- 对于步骤 1 和 2，维护 *m* 训练样本的数量并沿着每个编码的 128 个值求和。 `tf.reduce_sum` 有一个轴参数。这将选择沿哪个轴应用总和。

- 请注意，选择张量中最后一个轴的一种方法是使用负索引（axis=-1）。

- 在步骤 4 中，当对训练样本求和时，结果将是单个标量值。

- 要使 `tf.reduce_sum` 对所有轴求和，请保留默认值 axis=None。

In [ ]:
# UNQ_C1(UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: triplet_loss

def triplet_loss(y_true, y_pred, alpha = 0.2):
    """
    公式（3）定义的三元组损失的实现
    
    参数：
    y_true -- true labels，当你在Keras中定义损失时需要它，在这个函数中不需要它。
    y_pred -- 包含三个对象的 python 列表：
            anchor -- 锚点图像的编码，形状为 (None, 128)
            positive -- 正图像的编码，形状为 (None, 128)
            negative -- 负样本图像的编码，形状为 (None, 128)
    
    返回：
    loss -- 实数，损失值
    """
    
    anchor, positive, negative = y_pred[0], y_pred[1], y_pred[2]
    
    ### START CODE HERE
    #（≈ 4 行）
    # 步骤 1：计算锚点和正样本之间的（编码）距离
    pos_dist = None
    # 第 2 步：计算锚点样本与负样本之间的编码距离
    neg_dist = None
    # 第 3 步：将前面计算的两个距离相减，再加上 alpha。
    basic_loss = None
    # 步骤 4：取 basic_loss 和 0.0 中的最大值。对训练样本求和。
    loss = None
    ### END CODE HERE
    
    return loss

In [ ]:
### 你无法编辑此单元格

# 开始单元测试
tf.random.set_seed(1)
y_true = (None, None, None) # 未使用
y_pred = (tf.keras.backend.random_normal([3, 128], mean=6, stddev=0.1, seed = 1),
          tf.keras.backend.random_normal([3, 128], mean=1, stddev=1, seed = 1),
          tf.keras.backend.random_normal([3, 128], mean=3, stddev=4, seed = 1))
loss = triplet_loss(y_true, y_pred)

assert type(loss) == tf.python.framework.ops.EagerTensor, "Use tensorflow functions"
print("loss = " + str(loss))

y_pred_perfect = ([[1., 1.]], [[1., 1.]], [[1., 1.,]])
loss = triplet_loss(y_true, y_pred_perfect, 5)
assert loss == 5, "Wrong value. Did you add the alpha to basic_loss?"
y_pred_perfect = ([[1., 1.]],[[1., 1.]], [[0., 0.,]])
loss = triplet_loss(y_true, y_pred_perfect, 3)
assert loss == 1., "Wrong value. Check that pos_dist = 0 and neg_dist = 2 in this example"
y_pred_perfect = ([[1., 1.]],[[0., 0.]], [[1., 1.,]])
loss = triplet_loss(y_true, y_pred_perfect, 0)
assert loss == 2., "Wrong value. Check that pos_dist = 2 and neg_dist = 0 in this example"
y_pred_perfect = ([[0., 0.]],[[0., 0.]], [[0., 0.,]])
loss = triplet_loss(y_true, y_pred_perfect, -2)
assert loss == 0, "Wrong value. Are you taking the maximum between basic_loss and 0?"
y_pred_perfect = ([[1., 0.], [1., 0.]],[[1., 0.], [1., 0.]], [[0., 1.], [0., 1.]])
loss = triplet_loss(y_true, y_pred_perfect, 3)
assert loss == 2., "Wrong value. Are you applying tf.reduce_sum to get the loss?"
y_pred_perfect = ([[1., 1.], [2., 0.]], [[0., 3.], [1., 1.]], [[1., 0.], [0., 1.,]])
loss = triplet_loss(y_true, y_pred_perfect, 1)
if (loss == 4.):
    raise Exception('Perhaps you are not using axis=-1 in reduce_sum?')
assert loss == 5, "Wrong value. Check your implementation"

# 终端单元测试

**预期输出**：

<table>
    <tr>
        <td>
            <b>损失：</b>
        </td>
        <td>
           tf.Tensor(527.2598，形状=()，dtype=float32)
        </td>
    </tr>
    </table>

<a name='4'></a>
## 4 - 加载预训练模型

FaceNet 通过最小化三元组损失进行训练。由于训练需要大量数据和计算，这里无需从头训练；运行下面的单元格即可加载已训练好的模型，这可能需要几分钟。

In [ ]:
FRmodel = model

下面给出三个不同的人之间的人脸编码距离示例：

<img src="images/distance_matrix.png" style="width:380px;height:200px;"><br>
<caption><center> <u> <font color='purple'> <b>图 4：</b></u> <br> <font color='purple'>三个人的人脸编码之间的距离示例</center></caption>

现在使用该模型进行人脸验证和人脸识别！

<a name='5'></a>
## 5 - 应用模型

你正在为一座办公楼构建一个系统，大楼经理希望提供人脸识别功能以允许员工进入大楼。

你希望构建一个人脸验证系统，允许名单中的人员进入。每个人需要先在入口处刷员工身份卡，再由人脸识别系统验证其身份是否与所声明的身份一致。

<a name='5-1'></a>
### 5.1 - 人脸验证

接下来构建一个数据库，为每名获准进入办公室的人员保存一个编码向量。使用 `img_to_encoding(image_path, model)` 对指定图像执行模型的前向传播，生成编码。

运行下面的代码来建立数据库（用 Python 字典表示）。该数据库将每个人的姓名映射到其人脸的 128 维编码。

In [ ]:
#tf.keras.backend.set_image_data_format('channels_last')
def img_to_encoding(image_path, model):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(160, 160))
    img = np.around(np.array(img) / 255.0, decimals=12)
    x_train = np.expand_dims(img, axis=0)
    embedding = model.predict_on_batch(x_train)
    return embedding / np.linalg.norm(embedding, ord=2)

In [ ]:
database = {}
database["danielle"] = img_to_encoding("images/danielle.png", FRmodel)
database["younes"] = img_to_encoding("images/younes.jpg", FRmodel)
database["tian"] = img_to_encoding("images/tian.jpg", FRmodel)
database["andrew"] = img_to_encoding("images/andrew.jpg", FRmodel)
database["kian"] = img_to_encoding("images/kian.jpg", FRmodel)
database["dan"] = img_to_encoding("images/dan.jpg", FRmodel)
database["sebastiano"] = img_to_encoding("images/sebastiano.jpg", FRmodel)
database["bertrand"] = img_to_encoding("images/bertrand.jpg", FRmodel)
database["kevin"] = img_to_encoding("images/kevin.jpg", FRmodel)
database["felix"] = img_to_encoding("images/felix.jpg", FRmodel)
database["benoit"] = img_to_encoding("images/benoit.jpg", FRmodel)
database["arnaud"] = img_to_encoding("images/arnaud.jpg", FRmodel)

加载 Danielle 和 Kian 的图像： 

In [ ]:
danielle = tf.keras.preprocessing.image.load_img("images/danielle.png", target_size=(160, 160))
kian = tf.keras.preprocessing.image.load_img("images/kian.jpg", target_size=(160, 160))

In [ ]:
np.around(np.array(kian) / 255.0, decimals=12).shape

In [ ]:
kian

In [ ]:
np.around(np.array(danielle) / 255.0, decimals=12).shape

In [ ]:
danielle

现在，当有人出现在你的前门并刷员工身份卡（从而给你他们的名字）时，你可以在数据库中查找他们的编码，并用它来检查站在前门的人是否与员工身份卡上的名字匹配。

<a name='ex-2'></a>
### 练习 2 - verify

实现 `verify()` 函数，检查入口摄像头拍摄的图像（`image_path`）是否与 identity 指定的人员相符。请按以下步骤完成：

- 计算 `image_path` 中图像的编码。
- 计算该编码与数据库中对应身份的人脸图像编码之间的距离。
- 如果距离小于 0.7，就开门；否则不开门。

如上所述，应使用 L2 距离 `np.linalg.norm`。

**注意**：在此实现中，将 L2 距离（而不是 L2 距离的平方）与阈值 0.7 进行比较。

*提示*：

- `identity` 是一个字符串，也是数据库字典中的键。
- `img_to_encoding` 有两个参数：image_path 和 model。

In [ ]:
# UNQ_C2(UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: verify

def verify(image_path, identity, database, model):
    """
    函数，验证 "image_path" 图像上的人是否是 "identity"。
    
    参数：
        image_path -- 图像路径
        identity -- 字符串，你要验证身份的人的姓名。必须是在办公室工作的员工。
        database -- python 字典将允许的人员姓名（字符串）映射到其编码（向量）。
        model -- Keras 中的 Inception 模型实例
    
    返回：
        dist -- image_path 与数据库中 "identity" 图像之间的距离。
        door_open -- 正确，如果门应该打开。否则为假。
    """
    ### START CODE HERE
    # 步骤 1：计算图像的编码。使用 img_to_encoding() 参见上面的示例。 （≈ 1 行）
    encoding = None
    # 步骤 2：计算当前图像编码与 identity 对应人员图像编码之间的距离（约 1 行代码）
    dist = None
    # 步骤 3：如果 dist < 0.7 则开门，否则不开门（≈ 3 行）
    if None:
        print("It's " + str(identity) + ", welcome in!")
        door_open = None
    else:
        print("It's not " + str(identity) + ", please go away")
        door_open = None
    ### END CODE HERE        
    return dist, door_open

尤尼斯试图进入办公室，相机拍下了他的照片（“images/camera_0.jpg”）。让我们在这张图片上运行你的验证算法：

<img src="images/camera_0.jpg\" style="width:100px;height:100px;">

In [ ]:
### 你无法编辑此单元格

# 开始单元测试
distance, door_open_flag = verify("images/camera_0.jpg", "younes", database, FRmodel)
assert np.isclose(distance, 0.5992949), "Distance not as expected"
assert isinstance(door_open_flag, bool), "Door open flag should be a boolean"
print("(", distance, ",", door_open_flag, ")")
# 终端单元测试

**预期输出**：

<table>
    <tr>
        <td>
            <b>是你，欢迎光临！</b>
        </td>
        <td>
           （0.5992949，真实）
        </td>
    </tr>
    </table>

Benoit 并非办公室员工，却偷走了 Kian 的员工身份卡，试图进入办公室。摄像头拍下了 Benoit 的照片（images/camera_2.jpg）。

<img src="images/camera_2.jpg" style="width:100px;height:100px;">

运行验证算法检查Benoit是否可以进入。

In [ ]:
### 你无法编辑此单元格

verify("images/camera_2.jpg", "kian", database, FRmodel)

**预期输出**：

<table>
    <tr>
        <td>
            <b>这不是kian，请走开</b>
        </td>
        <td>
           （1.0259346，假）
        </td>
    </tr>
    </table>

<a name='5-2'></a>
### 5.2 - 人脸识别

你的人脸验证系统基本正常工作。但自从Kian的员工身份卡被盗后，第二天他回到办公室就进不去了！

为了解决这个问题，你想将人脸验证系统更改为人脸识别系统。这样一来，大家就不用再带员工身份卡了。授权人员只需走到大楼前，门就会为他们解锁！

你将实现一个人脸识别系统，该系统将图像作为输入，并确定它是否是授权人员之一（如果是，则是谁）。与之前的人脸验证系统不同，你将不再将人名作为输入之一。

<a name='ex-3'></a>
### 练习 3 - who_is_it

通过以下步骤实现 `who_is_it()`：

- 从 `image_path` 计算图像的目标编码
- 从数据库中查找与目标编码距离最小的编码。
- 将 `min_dist` 变量初始化为足够大的数字 (100)。这可以帮助你跟踪与输入编码最接近的编码。
- 循环数据库字典的名称和编码。要在 `database.items()` 中循环使用 for (name, db_enc)。
- 计算目标“编码”与数据库中当前“编码”之间的 L2 距离。如果此距离小于 min_dist，则将 min_dist 设置为 dist，并将 Identity 设置为 name。

In [ ]:
# UNQ_C3(UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: who_is_it

def who_is_it(image_path, database, model):
    """
    通过查找image_path图像上的人来实现办公室的人脸识别。
    
    参数：
        image_path -- 图像路径
        database -- 包含图像编码以及图像上人物姓名的数据库
        model -- Keras 中的 Inception 模型实例
    
    返回：
        min_dist -- image_path 编码与数据库编码之间的最小距离
        identity -- 字符串，image_path 上的人的名字预测
    """
    
    ### START CODE HERE

    ## 步骤1：计算图像的目标"encoding"。使用 img_to_encoding() 参见上面的示例。 ##（≈ 1 行）
    encoding =  None
    
    ## 步骤2：找到最接近的编码##
    
    # 将 "min_dist" 初始化为一个大值，比如 100（≈1 行）
    min_dist = None
    
    # 循环数据库字典的名称和编码。
    for (name, db_enc) in database.items():
        
        # 计算目标 "encoding" 和数据库中当前 db_enc 之间的 L2 距离。 （≈ 1 行）
        dist = None

        # 如果该距离小于 min_dist，则将 min_dist 设置为 dist，将 Identity 设置为 name。 （≈ 3 行）
        if None:
            min_dist = None
            identity = None
    ### END CODE HERE
    
    if min_dist > 0.7:
        print("Not in the database.")
    else:
        print ("it's " + str(identity) + ", the distance is " + str(min_dist))
        
    return min_dist, identity

尤尼斯在前门，相机拍摄了他的照片（“images/camera_0.jpg”）。让我们看看你的 `who_it_is()` 算法是否可以识别 Younes。

In [ ]:
### 你无法编辑此单元格

# 开始单元测试
# 测试 1（使用尤尼斯图片） 
who_is_it("images/camera_0.jpg", database, FRmodel)

# 测试 2（尤尼斯图片） 
test1 = who_is_it("images/camera_0.jpg", database, FRmodel)
assert np.isclose(test1[0], 0.5992946)
assert test1[1] == 'younes'

# 测试 3（尤尼斯图片） 
test2 = who_is_it("images/younes.jpg", database, FRmodel)
assert np.isclose(test2[0], 0.0)
assert test2[1] == 'younes'
# 终端单元测试

**预期输出**：

<table>
    <tr>
        <td>
            <b>it's younes，距离是</b> 0.5992949<br>
            <b>it's younes，距离是</b> 0.5992949<br>
            <b>it's younes，距离是</b> 0.0<br>
        </td>
    </tr>
    </table>

你可以将“camera_0.jpg”（Younes 的图片）更改为“camera_1.jpg”（Bertrand 的图片）并查看结果。

**恭喜**！ 
你已完成此作业，并且你的人脸识别系统运行良好！它不仅允许授权人员进入，而且现在人们不再需要携带员工身份卡！

你现在已经了解了最先进的人脸识别系统的工作原理，并且可以描述人脸识别和人脸验证之间的区别。以下是你已完成的工作的快速回顾： 

- 将人脸识别视为二元分类问题
- 针对人脸识别问题实现了一次性学习
- 应用三元组损失函数来学习人脸识别背景下的网络参数
- 使用预训练模型将人脸图像映射为 128 维编码
- 使用这些编码进行人脸验证和人脸识别

干得好！ 

<font color='blue'>
    
**你应该记住什么**：

- 人脸验证解决了更简单的1:1匹配问题；人脸识别解决了更困难的 1:K 匹配问题。
    
- 三元组损失是一种有效的损失函数，用于训练神经网络来学习人脸图像的编码。
    
- 可以使用相同的编码进行验证和识别。测量两个图像编码之间的距离可以让你确定它们是否是同一个人的照片。

**改进人脸识别模型的方法**：

虽然你不会在这里实现这些，但这里有一些进一步改进算法的方法：

- 将每个人的更多图像（在不同光照条件下、在不同日期拍摄等）放入数据库中。然后，给定一张新图像，将新面孔与该人的多张照片进行比较。这会提高准确性。

- 裁剪图像以仅包含脸部，并减少脸部周围的“边界”区域。这种预处理去除了脸部周围的一些不相关像素，也使算法更加鲁棒。

<a name='6'></a>
## 6 - 参考文献
1. 弗洛里安·施罗夫、德米特里·卡莱尼琴科、詹姆斯·菲尔宾 (2015)。 [FaceNet：人脸识别和聚类的统一嵌入](https://arxiv.org/pdf/1503.03832.pdf)

2. Yaniv Taigman、杨明、Marc'Aurelio Ranzato、Lior Wolf (2014)。 [DeepFace：缩小人脸验证性能与人类水平的差距](https://research.fb.com/wp-content/uploads/2016/11/deepface-closing-the-gap-to-human-level-performance-in-face-verification.pdf)

3. 这个实现也从FaceNet官方github存储库中获得了很多灵感：https://github.com/davidsandberg/facenet

4. 在这里找到了更多灵感：https://machinelearningmastery.com/how-to-develop-a-face-recognition-system-using-facenet-in-keras-and-an-svm-classifier/

5. 这里：https://github.com/nyoki-mtl/keras-facenet/blob/master/notebook/tf_to_keras.ipynb